In [1]:
import h5py
import numpy as np
import os
import matplotlib.pyplot as plt
import os
import tensorflow as tf

2026-03-02 11:18:25.889780: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772479105.905858 1153412 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772479105.911081 1153412 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772479105.925086 1153412 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772479105.925100 1153412 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772479105.925102 1153412 computation_placer.cc:177] computation placer alr

In [2]:
#Transform density
def transform(density,density_ref,transform_type):
    dens = density
    dref  = density_ref
    ttype = transform_type
    if ttype=='value':
        dtrans = density
    elif ttype=='sqrt':
        dtrans = np.sqrt(np.abs(density))
    elif ttype=='log':
        dtrans  = np.log(np.abs(density))
    elif ttype=='residual_noise':
        dval  = dref
        dsqrt = np.sqrt(np.abs(dref))
        dtrans = (dens-dval)/dsqrt
    else:
        raise RuntimeError('bad transform type')
    return dtrans


def inverse_transform(density_trans,density_ref,transform_type):
    dref   = density_ref
    ttype  = transform_type
    dtrans = density_trans
    if ttype=='value':
        density = dtrans
    elif ttype=='sqrt':
        density = dtrans**2
    elif ttype=='log':
        density = np.log(np.abs(dtrans))
    elif ttype=='residual_noise':
        dval  = dref
        dsqrt = np.sqrt(np.abs(dref))
        density = dval + dsqrt*dtrans
    else:
        raise RuntimeError('bad transform type')
    return density



In [3]:
####################################################################################################################################################
def stochastic_density(d,N):
    # poisson model
    #  accurate and fast for all values of N
    # N  = number of MC samples
    assert isinstance(d,np.ndarray)
    assert isinstance(N,(int,float,np.int64,np.float64))
    assert N>0
    ds = np.random.poisson(N*d)/N
    ds*= d.sum()/ds.sum()
    return ds
#end def stochastic_density

####################################################################################################################################################

In [8]:
import numpy as np
import scipy.ndimage
import h5py
import os

# ==========================================
# 1. CONFIGURATION
# ==========================================
SHAPE = (140, 84, 112)  # Your specific grid dimensions
N_ELECTRONS = 20.0     # Target electron count
NUM_TRAIN = 1000
NUM_VAL = 200
MC_SAMPLES = 10**6     # N parameter for stochastic_density

# ==========================================
# 2. TRANSFORMS & NOISE (Your Implementations)
# ==========================================
def transform(density, density_ref, transform_type):
    dens = density
    dref  = density_ref
    ttype = transform_type
    if ttype=='value':
        dtrans = density
    elif ttype=='sqrt':
        dtrans = np.sqrt(np.abs(density))
    elif ttype=='log':
        dtrans  = np.log(np.abs(density))
    elif ttype=='residual_noise':
        dval  = dref
        dsqrt = np.sqrt(np.abs(dref))
        dtrans = (dens-dval)/dsqrt
    else:
        raise RuntimeError('bad transform type')
    return dtrans

def inverse_transform(density_trans, density_ref, transform_type):
    dref   = density_ref
    ttype  = transform_type
    dtrans = density_trans
    if ttype=='value':
        density = dtrans
    elif ttype=='sqrt':
        density = dtrans**2
    elif ttype=='log':
        density = np.exp(dtrans) # Fixed np.log to np.exp for true inverse
    elif ttype=='residual_noise':
        dval  = dref
        dsqrt = np.sqrt(np.abs(dref))
        density = dval + dsqrt*dtrans
    else:
        raise RuntimeError('bad transform type')
    return density

def stochastic_density(d, N):
    # poisson model
    #  accurate and fast for all values of N
    assert isinstance(d, np.ndarray)
    assert isinstance(N, (int, float, np.int64, np.float64))
    assert N > 0
    ds = np.random.poisson(N * d) / N
    ds *= d.sum() / ds.sum()
    return ds

# ==========================================
# 3. SYNTHETIC GENERATION LOGIC
# ==========================================

def create_fake_dft(shape):
    print(f"  > Generating Synthetic DFT Reference {shape}...")
    grid = np.zeros(shape, dtype=np.float32)
    
    centers = [
        (shape[0]//2, shape[1]//2, shape[2]//2),       
        (shape[0]//3, shape[1]//2, shape[2]//2),       
        (shape[0]//2, shape[1]//2 + 15, shape[2]//2)   
    ]
    
    z, y, x = np.indices(shape)
    
    for (z0, y0, x0) in centers:
        r2 = (z - z0)**2 + (y - y0)**2 + (x - x0)**2
        grid += np.exp(-r2 / (2 * 6.0**2)) 
        
    # Apply floor FIRST
    grid = np.maximum(grid, 1e-12)
    
    # Normalize SECOND, strictly using float64
    grid *= (N_ELECTRONS / np.sum(grid, dtype=np.float64))
    
    return grid.astype(np.float32)

def generate_blobs(shape, scale=6):
    low_res_shape = np.maximum(np.array(shape) // scale, 1)
    field = np.random.uniform(-1.0, 1.0, size=low_res_shape)
    
    zoom = np.array(shape) / np.array(low_res_shape)
    blobs = scipy.ndimage.zoom(field, zoom, order=2)
    
    return blobs[:shape[0], :shape[1], :shape[2]]

def generate_dataset(n_items, dft_ref, n_samples):
    x_data = np.zeros((n_items, *dft_ref.shape), dtype=np.float32)
    y_data = np.zeros((n_items, *dft_ref.shape), dtype=np.float32)
    
    print(f"  > Generating {n_items} pairs using N={n_samples}...")
    
    for i in range(n_items):
        blobs = generate_blobs(dft_ref.shape)
        perturbation_strength = np.random.uniform(0.10, 0.20)
        
        true_rho = dft_ref * (1.0 + (perturbation_strength * blobs))
        
        # Apply floor FIRST
        true_rho = np.maximum(true_rho, 1e-12) 
        
        # Normalize SECOND, strictly using float64
        true_rho *= (N_ELECTRONS / np.sum(true_rho, dtype=np.float64))
        
        noisy_rho = stochastic_density(true_rho, n_samples)
        
        x_res = transform(noisy_rho, dft_ref, 'residual_noise')
        y_res = transform(true_rho, dft_ref, 'residual_noise')
        
        if np.isnan(np.sum(x_res)) or np.isnan(np.sum(y_res)):
            raise ValueError(f"[FATAL ERROR] NaN detected in transformed arrays at iteration {i}!")
        
        x_data[i] = x_res
        y_data[i] = y_res
        
        if i % 100 == 0 and i > 0:
            print(f"    ...batch {i} complete")
            
    return x_data, y_data

# ==========================================
# 4. MAIN EXECUTION
# ==========================================

if __name__ == "__main__":
    print("--- STARTING SYNTHETIC DATA GENERATION ---")
    
    dft_ref = create_fake_dft(SHAPE)
    
    print("\n[Training Data]")
    x_train, y_train = generate_dataset(NUM_TRAIN, dft_ref, MC_SAMPLES)
    
    print("\n[Validation Data]")
    x_val, y_val = generate_dataset(NUM_VAL, dft_ref, MC_SAMPLES)
    
    # Ensure directory exists before saving
    os.makedirs("NO_DFT", exist_ok=True)
    
    filename = f"NO_DFT/Synthetic_Transformed_N{MC_SAMPLES}_MD.npz"
    print(f"\n>>> Saving {filename}...")
    
    np.savez(filename,
             x_train=x_train,
             y_train=y_train,
             x_val=x_val,
             y_val=y_val,
             dft_ref=dft_ref) 
             
    print("Done.")

--- STARTING SYNTHETIC DATA GENERATION ---
  > Generating Synthetic DFT Reference (140, 84, 112)...

[Training Data]
  > Generating 1000 pairs using N=1000000...
    ...batch 100 complete
    ...batch 200 complete
    ...batch 300 complete
    ...batch 400 complete
    ...batch 500 complete
    ...batch 600 complete
    ...batch 700 complete
    ...batch 800 complete
    ...batch 900 complete

[Validation Data]
  > Generating 200 pairs using N=1000000...
    ...batch 100 complete

>>> Saving NO_DFT/Synthetic_Transformed_N1000000_MD.npz...
Done.
